# Agent的初步构建

主要使用工具 

    LangGraph, LangChain, Travily, 高德


In [1]:
import os 
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = "Agent with astream_event"
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY-API-KEY")
gaode_key = os.getenv("GAODEWEATHER_API_KEY")


创建工具并且进行包装

In [2]:
from langchain_tavily import TavilySearch
from langchain_core.tools import StructuredTool
from datetime import datetime
import requests

search_tool = TavilySearch(max_results=5)

def get_current_time(dummy: str = None) -> dict:
    "获取当前日期和时间"
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {"current_time": now}

time_tool = StructuredTool.from_function(
    func=get_current_time,
    name="get_current_time",
    description="如果你想知道对话发生的当前的日期和时间，请使用这个工具",
)


def get_adcode(keyword: str, api_key: str = gaode_key) -> dict:
    """
    根据地点关键词获取高德的 adcode
    参数:
        keyword: 地点名称，例如 "成都"
        api_key: 高德开放平台 API key
    返回:
        包含 adcode 的字典，例如 {"adcode": "510100"}
    """
    url = "https://restapi.amap.com/v3/config/district"
    params = {
        "key": api_key,
        "keywords": keyword,
        "subdistrict": 0  # 先不获取下级行政区
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        # 检查请求状态
        if data.get("status") == "1" and data.get("districts"):
            # 返回第一个匹配地区的adcode
            adcode = data["districts"][0]["adcode"]
            return {"adcode": adcode, "keyword": keyword}
        else:
            return {
                "error": data.get("info", "查询失败或未找到地区"),
                "keyword": keyword
            }
    except Exception as e:
        return {"error": str(e), "keyword": keyword}
    
adcode_tool = StructuredTool.from_function(
    func=get_adcode,
    name="get_adcode",
    description="用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询"
)


def get_weather_by_adcode(adcode: str, 
                          api_key: str = gaode_key, 
                          extensions: str ="all") -> dict:
    """
    根据高德行政区 adcode 查询天气信息。
    参数:
        adcode: 行政区划代码
        api_key: 高德开放平台的 API Key
        extensions: 
            - "base" 表示实况天气 
            - "all" 表示预报天气
    返回:
        包含天气信息的字典，例如:
        {"city": "成都", "weather": "多云", "temperature": "22", ...}
    """
    url = "https://restapi.amap.com/v3/weather/weatherInfo"
    params = {
        "key": api_key,
        "city": adcode,
        "extensions": extensions
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get("status") == "1":
            if extensions == "base":
                # 解析实况天气
                lives = data.get("lives", [])
                if lives:
                    live = lives[0]
                    return {
                        "province": live.get("province"),
                        "city": live.get("city"),
                        "weather": live.get("weather"),
                        "temperature": live.get("temperature"),
                        "winddirection": live.get("winddirection"),
                        "windpower": live.get("windpower"),
                        "humidity": live.get("humidity"),
                        "reporttime": live.get("reporttime")
                    }
            elif extensions == "all":
                # 解析预报天气
                forecasts = data.get("forecasts", [])
                if forecasts:
                    # 这里可以处理预报数据，通常会包含未来几天的天气
                    forecast = forecasts[0]
                    return forecast
            return {"error": "未找到天气数据", "adcode": adcode, "extensions": extensions}
        else:
            return {"error": data.get("info", "查询失败"), "adcode": adcode}
    except Exception as e:
        return {"error": str(e), "adcode": adcode}

get_weather_tool = StructuredTool.from_function(
    func=get_weather_by_adcode,
    name="get_weather_by_adcode",
    description="如果用户需要查询实时的天气，必须首先需要调用get_adcode工具获得adcode在调用"
)

In [3]:
tools = [time_tool, adcode_tool, get_weather_tool, search_tool]
tools

[StructuredTool(name='get_current_time', description='如果你想知道对话发生的当前的日期和时间，请使用这个工具', args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x000001F8A1F43880>),
 StructuredTool(name='get_adcode', description='用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询', args_schema=<class 'langchain_core.utils.pydantic.get_adcode'>, func=<function get_adcode at 0x000001F8D695D510>),
 StructuredTool(name='get_weather_by_adcode', description='如果用户需要查询实时的天气，必须首先需要调用get_adcode工具获得adcode在调用', args_schema=<class 'langchain_core.utils.pydantic.get_weather_by_adcode'>, func=<function get_weather_by_adcode at 0x000001F8D695D900>),
 TavilySearch(max_results=5, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [4]:
model = ChatDeepSeek(base_url=BASE_URL, api_key=API_KEY,
                     temperature=0.00,
                     model="deepseek-chat")

In [5]:
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, MessagesState, END
from langchain_core.messages import HumanMessage, AIMessage
from typing import Optional

class State(MessagesState):
    question: Optional[str] = None
    need_tools: Optional[bool] = None
    retry_count: Optional[str] = None
    tool_result: Optional[str] = None
    tool_result_history: Optional[list[str]] = None
    final_answer: Optional[str] = None

tool_usage_prompt = """
    你是一个擅长运用各种工具进行信息检索、数据分析和逻辑推理的专家。
    你的目标是提供准确、完整、有深度的答案。
    在开始之前，请先先确认这个问题是否需要使用工具。
    如果需要工具：规划为解决这个问题可能需要使用的工具，并简要说明选择每个工具的理由。
    使用对应的工具执行任务，最后把收集到的内容保持原样输出，并最后附上一句精简的总结。
    """
tool_agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=tool_usage_prompt
)

def decide_nedd_tool(state: State) -> State:
    
    question = state["messages"][-1].content

    decide_prompt = f"""
    你是一个智能工具决策助手。你有以下的工具：
    - get_current_time: 获取当前时间
    - get_adcode: 查询城市 adcode
    - get_weather_by_adcode: 查询天气（需先有 adcode）
    - TavilySearch: 通用搜索

    用户问题：{question}
    是否需要调用工具？只需回答 "tool" 或 "no_tool"。
    """
    response = model.invoke(decide_prompt)
    need_tools = "tool" in response.content.lower()
    return {**state,
            "need_tools": need_tools,
            "question": question
            }

def use_tool(state: State) -> State:
    # 初始化循环计数器
    retry_count = state.get("retry_count", 0) + 1
    question = state["question"] or state["messages"][-1].content

    # 调用 react agent
    mg = {"messages": [HumanMessage(content=state["question"])]}
    result = tool_agent.invoke(mg)

    # 保留历史记录
    last_message = result["messages"][-1]
    tool_result = last_message.content # if hasattr(last_message, "content") else str(last_message)
    
    # 更新状态
    tool_history = state.get("tool_result_history", []) + [tool_result]

    return {
        **state,
        "retry_count": retry_count,
        "tool_result": tool_result,
        "tool_result_history": tool_history,
        "messages": result["messages"]  # 把 agent 的完整消息历史合并
    }

def reflection(state: State) -> State:
    reflection_prompt = f"""
    用户问题：{state['question']}
    工具结果：{state.get('tool_result', '')}
    历史结果：{state.get('tool_result_history', [])}
    现在，请对你的初步答案进行审查，包括但不限于：
    1. 工具选择是否正确？是否反复调用？
    2. 是否需要重新调用工具？
    3. 如果需要，请给出改进建议。
    请判断是否需要重新调用工具。输出严格 JSON：
    {{
    "reflection_decision": "retry"/"next",
    "reflection_suggestion": "string"
    }}
    """
    # 检查重试次数，避免无限循环
    max_retries = 2  # 设置最大重试次数
    retry_count = state.get("retry_count", 0)
    if retry_count >= max_retries:
        return {**state, "reflection_decision": "next"}
    
    response = model.invoke(reflection_prompt)
    try:
        import json
        decision = json.loads(response.content)
    except:
        decision = {"reflection_decision": "next", "reflection_suggestion": "解析失败，跳过重试"}

    return {
        **state,
        "reflection_decision": decision["reflection_decision"],
        "reflection_suggestion": decision.get("reflection_suggestion", "")
    }


def summarize(state):
    summarize_prompt = f"""
    你是一名专业编辑，请根据以下信息回答用户问题。
    用户问题：{state["question"]}
    参考资料：{state.get("tool_result", "")}
    """

    final_msg:AIMessage = model.invoke(summarize_prompt)
    return {
        **state,
        "final_answer": final_msg.content,
        "messages": state["messages"] + [final_msg]  
    }

In [6]:
workflow = StateGraph(State)

workflow.add_node("decide", decide_nedd_tool)
workflow.add_node("tool", use_tool)
workflow.add_node("reflection", reflection)
workflow.add_node("summarize", summarize)

# 条件判断 decide -> tool/ summarize
def route_after_decide(state: State):
    return "tool" if state["need_tools"] else "summarize"
workflow.add_conditional_edges(
    'decide',
    route_after_decide,
    {"tool": "tool","summarize": "summarize"}
)

workflow.add_edge("tool", "reflection")

def route_after_reflection(state: State):
    return "tool" if state.get("reflection_decision") == "retry" else "summarize"
workflow.add_conditional_edges(
    'reflection',
    route_after_reflection,
    {"tool": "tool","summarize": "summarize"}
)

workflow.add_edge("summarize", END)
workflow.set_entry_point("decide")

my_agent = workflow.compile()

In [7]:
result = my_agent.invoke({
    "messages": [HumanMessage(content="我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。")]
})

# 最终答案在：
print(result["messages"][-1].content)
# 或
print(result["final_answer"])

根据明天的天气情况和汕头特色，为您整理了以下出行计划：

## 🌤️ 明日天气详情
**2025年10月12日（周日）**
- **白天**：多云，33°C，体感炎热
- **夜间**：阵雨，27°C  
- **风向**：东南风1-3级
- **特别提醒**：傍晚开始有阵雨，建议18:00前结束户外活动

## 🗺️ 推荐行程安排

### 上午（9:00-12:00）｜户外景点游览
**礐石风景区**（建议2-3小时）
- 乘坐索道俯瞰汕头全景
- 探访飘然亭、桃花涧等景点
- *温馨提示：天气炎热，建议携带饮用水*

### 中午（12:00-14:00）｜潮汕美食体验
**午餐推荐**：杏花吴记牛肉火锅
- 品尝地道潮汕牛肉火锅
- 避开正午高温时段

### 下午（14:00-17:30）｜室内外结合游览
**汕头小公园片区**（建议2-3小时）
- 游览民国风情建筑群
- 打卡百货大楼、邮局等历史建筑
- 品尝小公园老牌粽球
- *此时段可在室内外灵活安排*

### 傍晚（17:30-19:00）｜海滨休闲
**东海岸公园**（视天气情况调整）
- 如无下雨，可欣赏海滨日落
- 如已下雨，改为室内活动

### 晚上（19:00以后）｜美食探索
**晚餐选择**：
- 潮汕生腌 + 卤水鹅
- 或品尝地道粿条汤
- *推荐：日日香卤鹅饭店*

## 🎒 出行必备物品
- **雨具**：折叠伞或雨衣（傍晚必带）
- **防晒**：防晒霜、遮阳帽、太阳镜
- **衣物**：透气夏装，备件薄外套
- **其他**：充电宝、饮用水、少量现金

## ⚠️ 重要提醒
1. **防雨准备**：阵雨可能突然来临，雨具随身携带
2. **防暑措施**：白天高温，及时补充水分，避免中暑
3. **交通建议**：市内景点间可乘坐公交或网约车
4. **美食提示**：生腌海鲜初次尝试需适量

## 💎 总结建议
明天汕头天气总体适宜出行，白天炎热需防晒，傍晚阵雨要防雨。建议重点体验潮汕美食文化，合理安排室内外活动时间，享受愉快的汕头之旅！

祝您旅途愉快！如有其他需求，欢迎随时咨询。
根据明天的天气情况和汕头特色，为您整理了以下出行计划：

## 🌤️ 明日天气详情
**2025年10月12日（周日）**
- **白天**：多云，33°C，体感炎热
- **夜间**：阵雨，

In [8]:
for item in result["messages"]:
    item.pretty_print()

================================ Human Message =================================

我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。
================================ Human Message =================================

我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。
================================== Ai Message ==================================

我来帮您查询汕头的天气情况并制定出行计划。首先让我获取汕头的地理编码，然后查询天气信息。
Tool Calls:
  get_adcode (call_00_QxjLX3GwwTZYR6sVmLCeU7uz)
 Call ID: call_00_QxjLX3GwwTZYR6sVmLCeU7uz
  Args:
    keyword: 汕头
================================= Tool Message =================================
Name: get_adcode

{"adcode": "440500", "keyword": "汕头"}
================================== Ai Message ==================================

现在让我查询汕头明天的详细天气情况：
Tool Calls:
  get_weather_by_adcode (call_00_htrMbkApiuukSmiVFAuI7iWd)
 Call ID: call_00_htrMbkApiuukSmiVFAuI7iWd
  Args:
    adcode: 440500
    extensions: all
================================= Tool Message =================================
Name: get_weather_by_adcode

{"city": 

print


In [ ]:
def my_print(event):
    item = None
    
    if "decide" in event:
        print("模型正在判断是否进行工具调用")
        if "need_tools" in event["decide"]:
            for k in event["decide"]["messages"]:
                k.pretty_print()
            # print("\n用户：", event["decide"]["question"])
            # print("\n工具调用决策：", event["decide"]["need_tools"])

    if "tool" in event:
        print("模型正在使用工具...")
        if "tool_result" in event["tool"]:
            for k in event["tool"]["messages"]:
                k.pretty_print()
            # print("\n模型检索结果/资料：", event["tool"]["tool_result"])
    
    if "summarize" in event:
        print("正在总结🧭")
        # print(event["summarize"].keys())
        item = event["summarize"]
        if "final_answer" in event["summarize"]:
            for k in event["summarize"]["messages"]:
                k.pretty_print()
            # print("\n 最终总结：", event["summarize"]["final_answer"])
    return item

In [45]:
messages = HumanMessage(content="我今天晚上9点到新疆，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？")
for event in my_agent.stream({"messages": [messages]}):
    my_print(event)

模型正在判断是否进行工具调用
================================ Human Message =================================

我今天晚上9点到新疆，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？
模型正在使用工具...
================================ Human Message =================================

我今天晚上9点到新疆，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？
================================== Ai Message ==================================

我来帮您查询新疆当地的情况和注意事项。首先让我获取当前时间，然后查询新疆的天气情况，最后搜索相关的风俗习惯和注意事项。
Tool Calls:
  get_current_time (call_00_CKZXHmYI5vRjUkzSrGYO9Tuy)
 Call ID: call_00_CKZXHmYI5vRjUkzSrGYO9Tuy
  Args:
    dummy: None
================================= Tool Message =================================
Name: get_current_time

{"current_time": "2025-10-11 18:30:23"}
================================== Ai Message ==================================

现在让我查询新疆的adcode，然后获取天气信息：
Tool Calls:
  get_adcode (call_00_7Uzvn9qF36MKt6SNWlKpxxJO)
 Call ID: call_00_7Uzvn9qF36MKt6SNWlKpxxJO
  Args:
    keyword: 新疆
================================= Tool Message =====================

In [47]:
print(item.content)

根据明天的天气情况和汕头特色，为您整理了以下出行计划：

## 🌤️ 明日天气详情
**2025年10月12日（周日）**
- **白天**：多云，33°C，体感炎热
- **夜间**：阵雨，27°C  
- **风向**：东南风1-3级
- **特别提醒**：傍晚开始有阵雨，建议18:00前结束户外活动

## 🗺️ 推荐行程安排

### 上午（9:00-12:00）｜户外景点游览
**礐石风景区**（建议2-3小时）
- 乘坐索道俯瞰汕头全景
- 探访飘然亭、桃花涧等景点
- *温馨提示：天气炎热，建议携带饮用水*

### 中午（12:00-14:00）｜潮汕美食体验
**午餐推荐**：杏花吴记牛肉火锅
- 品尝地道潮汕牛肉火锅
- 避开正午高温时段

### 下午（14:00-17:30）｜室内外结合游览
**汕头小公园片区**（建议2-3小时）
- 游览民国风情建筑群
- 打卡百货大楼、邮局等历史建筑
- 品尝小公园老牌粽球
- *此时段可在室内外灵活安排*

### 傍晚（17:30-19:00）｜海滨休闲
**东海岸公园**（视天气情况调整）
- 如无下雨，可欣赏海滨日落
- 如已下雨，改为室内活动

### 晚上（19:00以后）｜美食探索
**晚餐选择**：
- 潮汕生腌 + 卤水鹅
- 或品尝地道粿条汤
- *推荐：日日香卤鹅饭店*

## 🎒 出行必备物品
- **雨具**：折叠伞或雨衣（傍晚必带）
- **防晒**：防晒霜、遮阳帽、太阳镜
- **衣物**：透气夏装，备件薄外套
- **其他**：充电宝、饮用水、少量现金

## ⚠️ 重要提醒
1. **防雨准备**：阵雨可能突然来临，雨具随身携带
2. **防暑措施**：白天高温，及时补充水分，避免中暑
3. **交通建议**：市内景点间可乘坐公交或网约车
4. **美食提示**：生腌海鲜初次尝试需适量

## 💎 总结建议
明天汕头天气总体适宜出行，白天炎热需防晒，傍晚阵雨要防雨。建议重点体验潮汕美食文化，合理安排室内外活动时间，享受愉快的汕头之旅！

祝您旅途愉快！如有其他需求，欢迎随时咨询。
